# Localization, one niche at a time, through the harness


**What this is.** The first localization scores on a frozen partition (instruction 70). Localization is the target Einar chose first; the harness scores yes/no questions, so each coarse niche is one question.

**The labels.** The 718 final hyperLOPIT markers of Barylyuk et al. 2020 (mmc5, sha256 `a58a8778...`), merged from 26 niches to coarse ones (rhoptries 1 and 2 are rhoptries). For a niche, its markers are "yes" and every other marker is "no" -- a curated negative, not an unlabelled gene. **Caveat, stated first:** the paper chose 656 of the markers from prior knowledge, inferred function and profile consistency, so marker status is curation, not independent experiment, and evidence that encodes prior knowledge may recover it.

**What is held out.** Every column of the `place` category, whatever its family -- the hyperLOPIT experiment, its posteriors, and transfers built from it.

Built 2026-11-06.

In [1]:
import os
import numpy as np, pandas as pd
from parascape import harness as H, datasets as ds, organisms as O
from parascape.question_registry import Question
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
m = pd.read_excel(MARKERS, 'S4a - TAGM-MAP classification')
m = m[m.markers.astype(str) != 'unknown'].set_index('Accession')
COARSE = {'rhoptries 1': 'rhoptries', 'rhoptries 2': 'rhoptries', 'apical 1': 'apical', 'apical 2': 'apical',
          'ER 1': 'ER', 'ER 2': 'ER', 'PM - peripheral 1': 'plasma membrane', 'PM - peripheral 2': 'plasma membrane',
          'PM - integral': 'plasma membrane', 'nucleus - chromatin': 'nucleus', 'nucleus - non-chromatin': 'nucleus',
          'nucleolus': 'nucleus', 'mitochondrion - membranes': 'mitochondrion', 'mitochondrion - soluble': 'mitochondrion',
          '40S ribosome': 'ribosome', '60S ribosome': 'ribosome', '19S proteasome': 'proteasome', '20S proteasome': 'proteasome'}
niche = m.markers.astype(str).map(lambda v: COARSE.get(v, v))
niche = niche[niche.index.isin(nodes.index)]
counts = niche.value_counts()
print(len(m), 'markers;', len(niche), 'in the node table;', counts.size, 'coarse niches')
counts.to_frame('markers')

718 markers; 716 in the node table; 16 coarse niches


,markers
markers,
nucleus,109
mitochondrion,68
plasma membrane,66
cytosol,62
ribosome,59
apicoplast,48
rhoptries,47
apical,41
dense granules,40


## The questions, and the evidence each may use

Coarse niches with at least 25 markers. Each question holds out the `place` category.

In [2]:
def family_of(column):
    found = ds.provenance(column, O.TOXOPLASMA)
    return found.key if found is not None else 'unregistered'
numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c]) or pd.api.types.is_bool_dtype(nodes[c])]
families = pd.Series({c: family_of(c) for c in numeric}, name='family')
full = nodes.loc[niche.index, numeric].astype('float64')
groups = nodes['orthogroup'].reindex(niche.index)
groups = groups.where(groups.notna(), pd.Series({g: 'self:' + g for g in niche.index}))
NICHES = [n for n, k in counts.items() if k >= 25]
def question(n):
    return Question(id='localization_' + n.replace(' ', '_'), text=f'Is this protein in the {n}?',
                    answer_type='boolean', question_class='B', label_source='barylyuk2020 mmc5 markers',
                    label_family='lopit_tgon', label_tier='observed-curated', label_category='place',
                    evidence_forbidden=('lopit_tgon',))
features = H.allowed_features(question(NICHES[0]), full, families)
print(len(NICHES), 'questions;', features.shape[1], 'allowed columns of', full.shape[1],
      '(', full.shape[1] - features.shape[1], 'held out as place )')

13 questions; 373 allowed columns of 411 ( 38 held out as place )


## The runs

Boosted trees over every allowed column, nested 3 x 3 cross-validation inside train + tune, calibration, then the frozen partition once per question (seed 0, logged).

In [3]:
rows = []
for n in NICHES:
    q = question(n)
    y = (niche == n)
    r = H.run_question(q, labels=y, features=features, families=families, groups=groups,
                       strategies=[H.Strategy.of('boosted trees', 'hist_gbt', max_iter=150)], seed=0,
                       importance_repeats=2, log_path=os.path.join(REPO, H.HOLDOUT_LOG))
    cv, ft = r.cross_validated, r.frozen_test or {}
    base = r.baselines[r.baselines.partition == 'frozen_test']
    top = r.importance.head(3).family.tolist() if len(r.importance) else []
    rows.append(dict(niche=n, markers=int(y.sum()), cv_auc=round(cv['roc_auc'], 3), cv_ap=round(cv['average_precision'], 3),
                     test_n=ft.get('n'), test_positives=ft.get('positives'), test_auc=round(ft.get('roc_auc', np.nan), 3),
                     test_ap=round(ft.get('average_precision', np.nan), 3), test_calls=ft.get('calls'),
                     test_precision=ft.get('precision'), test_lower90=round(ft.get('precision_lower90', np.nan), 3),
                     best_baseline_lower90=round(float(base.precision_lower90.max()), 3) if len(base) else np.nan,
                     may_fill=H.may_fill(r), top_families=', '.join(top), flags=';'.join(r.flags)))
scores = pd.DataFrame(rows).sort_values('test_auc', ascending=False)
scores.to_csv(os.path.join(OUT, 'scores.tsv'), sep='\t', index=False)
scores

,niche,markers,cv_auc,cv_ap,test_n,test_positives,test_auc,test_ap,test_calls,test_precision,test_lower90,best_baseline_lower90,may_fill,top_families,flags
4,ribosome,59,0.998,0.96,104,10,1,1,12,0.8333,0.614,0.562,False,"gse51780, alphafold, bioid_corpus_membership",frozen_test_repeat
12,proteasome,28,0.978,0.716,104,4,1,1,1,1,0.1,0.021,False,"gse108740, gse19092, bioid_corpus_membership",frozen_test_repeat
5,apicoplast,48,0.927,0.58,104,6,0.997,0.958,3,1,0.464,0.013,False,"gse19092, gse22258, ipms_corpus_membership",frozen_test_repeat
8,dense granules,40,0.957,0.772,104,9,0.993,0.94,0,NaN,NaN,0,False,"gra12, gse19092, alphafold",frozen_test_repeat
6,rhoptries,47,0.974,0.82,104,7,0.99,0.844,2,1,0.316,0.631,False,"gse19092, gra12, toxodb_enteroepithelial",frozen_test_repeat
11,micronemes,29,0.932,0.431,104,4,0.985,0.692,2,0.5,0.051,0.154,False,"gse19092, unregistered, gse168465",frozen_test_repeat
0,nucleus,109,0.94,0.818,104,12,0.964,0.736,4,0.75,0.32,0.007,False,"gse19092, unregistered, proteome_total",frozen_test_repeat
7,apical,41,0.943,0.718,104,4,0.96,0.8,2,1,0.316,0.017,False,"gse19092, melting_temperature, proteome_total",frozen_test_repeat
3,cytosol,62,0.913,0.675,104,10,0.933,0.643,4,0.75,0.32,0,False,"proteome_total, unregistered, alphafold",frozen_test_repeat
1,mitochondrion,68,0.844,0.437,104,13,0.925,0.685,2,1,0.316,0,False,"gse19092, unregistered, foldseek_struct",frozen_test_repeat


## Sensitivity: genome-wide evidence only

Some evidence families were measured on a chosen subset of genes -- `gra12` is a CRISPR screen of 236 genes, a library of secretory candidates. Whether a gene was measured at all then encodes the choice someone made, which is prior knowledge of location by another door: the trees can read the missingness, not the measurement. The same questions are run again with only the families measured on at least half the genes, on a fresh seed (a different strategy may not look at the first partition). A niche that holds up here is placed by measurement; one that collapses was placed by who chose to measure it.

In [4]:
coverage = nodes[numeric].notna().mean()
family_cover = coverage.groupby(families.reindex(numeric).values).max()
WIDE = sorted(f for f, c in family_cover.items() if c >= 0.5 and f in set(families.reindex(features.columns)))
print(len(WIDE), 'genome-wide families of', families.reindex(features.columns).nunique())
rows = []
for n in NICHES:
    q = question(n)
    y = (niche == n)
    r = H.run_question(q, labels=y, features=features, families=families, groups=groups,
                       strategies=[H.Strategy.of('boosted trees, genome-wide families', 'hist_gbt', families=WIDE, max_iter=150)],
                       seed=1, importance_repeats=2, log_path=os.path.join(REPO, H.HOLDOUT_LOG))
    cv, ft = r.cross_validated, r.frozen_test or {}
    rows.append(dict(niche=n, wide_cv_auc=round(cv['roc_auc'], 3), wide_test_auc=round(ft.get('roc_auc', np.nan), 3),
                     wide_top_families=', '.join(r.importance.head(3).family.tolist()) if len(r.importance) else ''))
wide = scores[['niche', 'cv_auc', 'test_auc']].merge(pd.DataFrame(rows), on='niche')
wide['cv_change'] = (wide.wide_cv_auc - wide.cv_auc).round(3)
wide.to_csv(os.path.join(OUT, 'genome_wide_sensitivity.tsv'), sep='\t', index=False)
wide

50 genome-wide families of 85


,niche,cv_auc,test_auc,wide_cv_auc,wide_test_auc,wide_top_families,cv_change
0,ribosome,0.998,1,0.993,0.997,"gse51780, alphafold, bioid_corpus_membership",-0.005
1,proteasome,0.978,1,0.96,1,"gse19092, gse108740, toxodb_enteroepithelial",-0.018
2,apicoplast,0.927,0.997,0.923,0.956,"gse19092, crispr_serum_restriction, gse245775",-0.004
3,dense granules,0.957,0.993,0.937,0.928,"interpro, gse19092, gse200962",-0.02
4,rhoptries,0.974,0.99,0.977,0.968,"gse19092, toxodb_enteroepithelial, crispr_parasite_density",0.003
5,micronemes,0.932,0.985,0.959,0.95,"gse19092, gse168465, gse108740",0.027
6,nucleus,0.94,0.964,0.9,0.938,"gse19092, unregistered, gse245775",-0.04
7,apical,0.943,0.96,0.966,0.934,"gse19092, unregistered, crispr_parasite_density",0.023
8,cytosol,0.913,0.933,0.915,0.912,"unregistered, alphafold, toxodb_ec_numbers",0.002
9,mitochondrion,0.844,0.925,0.844,0.799,"foldseek_struct, unregistered, gse19092",0


## Reading it

AUC says whether the evidence ranks a niche's proteins above the other markers; the frozen-test lower bound says whether any calls are good enough to write into the map. With a handful of positives per frozen partition the bound is wide by construction -- more markers, not a better model, is what tightens it. The top families show what carries each niche; a family that restates prior knowledge (a curated annotation) is the caveat above, showing.

In [5]:
summary = pd.DataFrame([dict(questions=len(scores), median_test_auc=float(scores.test_auc.median()),
    above_0_8=int((scores.test_auc >= 0.8).sum()), may_fill=int(scores.may_fill.sum()),
    median_genome_wide_cv_auc=float(wide.wide_cv_auc.median()), median_cv_change=float(wide.cv_change.median()))])
summary.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t', index=False)
summary

,questions,median_test_auc,above_0_8,may_fill,median_genome_wide_cv_auc,median_cv_change
0,13,0.964,13,0,0.928,0
